In [ ]:
from google.cloud import pubsub_v1
import json
import os

# 1. Configuration from your .env
project_id = os.getenv("GCP_PROJECT_ID")
subscription_id = "bitcoin-price-sub" # The name from your Terraform

# 2. Initialize the Subscriber
subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(project_id, subscription_id)

print(f"🚀 Listening for Bitcoin data on {subscription_id}...")

def callback(message: pubsub_v1.subscriber.message.Message) -> None:
    """
    Process the incoming Bitcoin price message.
    """
    # Parse the data
    payload = json.loads(message.data.decode("utf-8"))
    
    # Senior View: Extract key metrics
    asset = payload['data']['asset']
    price_aud = payload['data']['price_aud']
    timestamp = payload['metadata']['ingested_at']
    
    print(f"✅ Received {asset.upper()}: ${price_aud:,.2f} AUD | Ingested: {timestamp}")
    
    # CRITICAL: Acknowledge the message so it's removed from the 'Unacked' graph
    message.ack()

# 3. Start the "Pull" flow
# We limit to 5 messages for this notebook test
streaming_pull_future = subscriber.subscribe(subscription_path, callback=callback)

try:
    # Keep the notebook cell alive for 30 seconds to catch messages
    streaming_pull_future.result(timeout=30)
except Exception as e:
    streaming_pull_future.cancel()
    print(f"\n⏹️ Stopped listening. (Note: {e})")

🚀 Listening for Bitcoin data on bitcoin-price-sub...
✅ Received ETHEREUM: $2,962.26 AUD | Ingested: 2026-03-06T03:42:45.067960+00:00
✅ Received ETHEREUM: $2,963.33 AUD | Ingested: 2026-03-06T03:41:36.529500+00:00
✅ Received BITCOIN: $101,058.00 AUD | Ingested: 2026-03-06T03:41:32.398735+00:00
✅ Received BITCOIN: $101,039.00 AUD | Ingested: 2026-03-06T03:42:36.976605+00:00


KeyboardInterrupt: 

Message (id=18573316044688897, ack_id=C0RbSDofGScFTF5FLTg1aDwFUUZTBwcrHUQSL0UADyhSDVcEexJhdWpaEwALFgB_WFkSWGpdVnUNUSUpaoXk7JdsMSADTVZ5XFgaC2hYWHMPWQUQdVWOiL3xo5zKaklCZ6OelJVqfIWEq58rZi89IBJLLD5-KTkHRl1YWgEhHQwEUTsBCSldTl9HMWI2KjUaUBxRGQw7, ordering_key=, exactly_once=False)'s callback threw exception, nacking message.
Traceback (most recent call last):
  File "/home/wooliterchen/miniconda3/envs/woolie-platform-env/lib/python3.13/site-packages/google/cloud/pubsub_v1/subscriber/_protocol/streaming_pull_manager.py", line 177, in _wrap_callback_errors
    callback(message)
    ~~~~~~~~^^^^^^^^^
  File "/tmp/ipykernel_8844/1854295971.py", line 23, in callback
    asset = payload['data']['asset']
            ~~~~~~~^^^^^^^^
KeyError: 'data'
Message (id=18573316044688897, ack_id=OwtEW0g6HxknBUxeRS04NWg8BVFGUwcHKx1EEi9FAA8oUg1XBHsSYXVqWhMACxYAf1hZElhqXVZ1DVElKWqE5OyXbDEgA01WeVxYGgtoWFhzD1kFEHVVjoi98aOcympJRWeinpSVanydgsQJZi89IBJLLD5-KTkHRl1YWgEhHQwEUTsBCSldTl9HMWI2KjUaUBxRGQw, ordering_key=, e